# Paper 06 · Auto-Encoding Variational Bayes

**Citation:** Diederik P. Kingma, Max Welling, “Auto-Encoding Variational Bayes” (2013).

**Paper:** https://arxiv.org/abs/1312.6114

> **Scale gap:** We train a tiny VAE on 8×8 digits, not large image benchmarks.

## Before you read
1. Why can't we naively backpropagate through a random sample?
2. What roles do reconstruction and KL terms play?
3. What would happen if the KL weight were zero?

## Central claim
The reparameterization trick makes stochastic latent-variable models trainable with standard gradient-based optimization.

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from torch import nn
from sklearn.datasets import load_digits
d=load_digits()
X=torch.tensor((d.data/16).astype("float32"))
torch.manual_seed(0)

## VAE with explicit reparameterization

In [ ]:
class VAE(nn.Module):
    def __init__(self,zdim=2):
        super().__init__()
        self.enc=nn.Sequential(nn.Linear(64,32),nn.ReLU())
        self.mu=nn.Linear(32,zdim); self.logvar=nn.Linear(32,zdim)
        self.dec=nn.Sequential(nn.Linear(zdim,32),nn.ReLU(),nn.Linear(32,64),nn.Sigmoid())
    def forward(self,x):
        h=self.enc(x); mu=self.mu(h); logvar=self.logvar(h)
        eps=torch.randn_like(mu)
        # TODO: after the first run, write the reparameterization line from memory.
        z=mu+torch.exp(.5*logvar)*eps
        return self.dec(z),mu,logvar,z

def train(beta=1.0,steps=350):
    m=VAE(); opt=torch.optim.Adam(m.parameters(),lr=.01); hist=[]
    for _ in range(steps):
        rec,mu,lv,z=m(X)
        recon=((rec-X)**2).mean()
        kl=(-.5*(1+lv-mu.pow(2)-lv.exp()).sum(1)).mean()/64
        loss=recon+beta*kl
        opt.zero_grad(); loss.backward(); opt.step(); hist.append([loss.item(),recon.item(),kl.item()])
    return m,np.array(hist)

## Reproduce latent regularization behavior

In [ ]:
m1,h1=train(beta=1.0)
m0,h0=train(beta=0.0)
for name,h in [("VAE beta=1",h1),("No KL",h0)]:
    print(name,"final loss/recon/KL",h[-1])
plt.plot(h1[:,1],label="VAE recon"); plt.plot(h0[:,1],label="no-KL recon"); plt.legend(); plt.show()
with torch.no_grad():
    _,mu,_,_=m1(X)
    Z=mu.numpy()
plt.scatter(Z[:,0],Z[:,1],c=d.target,s=5,cmap="tab10"); plt.title("2-D VAE latent means"); plt.show()

### Ablation
Sweep `beta` across 0, 0.1, 1, and 4. Compare reconstruction error, KL, and latent-space organization.

## Ablation table
Fill this after running the experiments.

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
Answer without looking back at the notebook:
1. What problem existed before this work?
2. What was actually new?
3. What evidence in your reproduction supports the central claim?
4. What does your reduced-scale reproduction **not** establish?
5. Which idea from this paper survived into modern systems?
6. What experiment would you run next?